<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [598]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'champs_elysees.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [599]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 1579 entries, Unnamed: 0 to lag_or_35_23
dtypes: float64(1564), int64(1), object(14)
memory usage: 116.2+ MB


In [600]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 
    'force moyenne vent (m/s)', 'jour_semaine', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton','est_rentree'
]

target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)','est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton', 'est_rentree'
]

categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=18,
    max_iter=70,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
W_RENTREE = 10
POST_SCALE_FERIE = 1.0

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()
    is_rentree = X_frame['est_rentree'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON
    w[is_rentree == 1] = W_RENTREE

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

df_final['Débit_prédit'] = np.nan
df_final.loc[X_test.index, 'Débit_prédit'] = y_pred


R² : 0.786
MAE : 82.97
RMSE : 116.85
Part d'observations piéton (test) : 2.1%
MAE (jours piéton) : 157.33 (n=34)
MAE (jours non piéton) : 81.40 (n=1606)


In [601]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 1580 entries, Unnamed: 0 to Débit_prédit
dtypes: datetime64[ns, UTC](1), float64(1565), int64(1), object(13)
memory usage: 116.3+ MB


In [602]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
2                  heure_sin       173.467777        3.467998
3                  heure_cos       131.280951        4.439839
4                   jour_sin        45.634136        2.540461
15                est_pieton        12.329409        1.073341
10               est_weekend         4.921097        0.965806
8   force moyenne vent (m/s)         3.514321        0.398295
13                 est_ferie         3.279161        2.039756
5                   jour_cos         3.123326        0.428111
9               jour_semaine         2.405709        0.410596
0                Température         2.260591        0.548441
12        est_avant_vacances         0.036174        0.031122
16               est_rentree         0.000000        0.000000
1       precipitations heure        -0.021875        0.110183
14           est_avant_ferie        -0.051925        0.115665
11              est_vacances        -0.192825        0.667971
7       

In [603]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [604]:
X_all = df_final.loc[:, features].copy()

predictions_all = pipe.predict(X_all)

serie = pd.Series(predictions_all, index=df_final.index)

commun = df_final[target].notna() & serie.notna()

predictions_with_know = serie.loc[commun].astype(float)

mae = mean_absolute_error(predictions_with_know, y_known)
rmse = np.sqrt(mean_squared_error(predictions_with_know, y_known))
r2   = r2_score(predictions_with_know, y_known)

print("Nombre de valeurs : " + str(len(predictions_with_know)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

df_final['débit_prédit_all'] = predictions_all 


Nombre de valeurs : 8197
R² : 0.800
MAE : 71.47
RMSE : 114.49


In [605]:
df_final['débit_prédit_all'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 9644 entries, 0 to 9643
Series name: débit_prédit_all
Non-Null Count  Dtype  
--------------  -----  
9644 non-null   float64
dtypes: float64(1)
memory usage: 75.5 KB


In [606]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final['Débit horaire'],
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

<h2>Taux d'occupation</h2>

In [607]:
target_occ = 'Taux d\'occupation'

features_occ = [
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin',  
    'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton', 'lag_or_4_0', 'lag_or_4_1', 'lag_or_5_0', 'lag_or_5_1', 'lag_or_4_23',
    'lag_or_6_23', 'lag_or_7_0', 'lag_or_7_1', 'est_rentree'
]

mask_occ = df_final[target_occ].notna()
mask_missing_occ = df_final[target_occ].isna()

X_occ = df_final.loc[mask_occ, features_occ].copy()
y_occ = df_final.loc[mask_occ, target_occ].astype(float)

X_train_occ, X_test_occ, y_train_occ, y_test_occ = train_test_split(
    X_occ, y_occ, test_size=0.18, shuffle=False
)

numeric_features_occ = [c for c in features_occ if c != 'jour_semaine']
categorical_features_occ = []

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_occ = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features_occ),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features_occ),
    ],
    remainder='drop'
)

model_occ = HistGradientBoostingRegressor(
    loss='absolute_error',
    max_depth=15,
    max_iter=200,
    early_stopping=False,
    random_state=42
)

pipe_occ = Pipeline(steps=[('prep', preprocess_occ), ('model', model_occ)])

pipe_occ.fit(X_train_occ, y_train_occ)

y_pred_occ = pipe_occ.predict(X_test_occ)

print("=== Performances taux d'occupation ===")
print(f"R²   : {r2_score(y_test_occ, y_pred_occ):.3f}")
print(f"MAE  : {mean_absolute_error(y_test_occ, y_pred_occ):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_occ, y_pred_occ)):.2f}")
print(f"nb test : {len(y_test_occ)}")


=== Performances taux d'occupation ===
R²   : 0.724
MAE  : 2.90
RMSE : 4.63
nb test : 1473


In [608]:
perm = permutation_importance(
    estimator=pipe_occ,
    X=X_test_occ,
    y=y_test_occ,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  
)

imp_df = (
    pd.DataFrame({
        'feature': features_occ,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


               feature  importance_mean  importance_std
0            heure_sin         4.136108        0.138648
1            heure_cos         0.791533        0.047062
17          lag_or_7_0         0.477161        0.044538
2             jour_sin         0.226932        0.039308
16         lag_or_6_23         0.155939        0.019233
18          lag_or_7_1         0.078032        0.017096
10          est_pieton         0.073447        0.018870
15         lag_or_4_23         0.051170        0.018185
6         est_vacances         0.042981        0.018386
11          lag_or_4_0         0.029434        0.011696
5          est_weekend         0.029241        0.011197
3             jour_cos         0.026729        0.016797
13          lag_or_5_0         0.018968        0.008526
8            est_ferie         0.014728        0.007430
9      est_avant_ferie         0.001593        0.002152
19         est_rentree         0.000000        0.000000
14          lag_or_5_1        -0.000032        0

In [609]:
time_index_occ = df_final.loc[X_test_occ.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_pred_occ,
    mode='lines',
    name="Taux d'occupation prédit"
))
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_test_occ,
    mode='lines',
    name= "Taux d'occupation réel"
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Taux d'occupation (%)",
    hovermode='x unified'
)

fig.show()

In [610]:
X_all_occ = df_final.loc[:, features_occ].copy()

predictions_all_occ = pipe_occ.predict(X_all_occ)

serie_occ = pd.Series(predictions_all_occ, index=df_final.index)

commun_occ = df_final[target_occ].notna() & serie_occ.notna()

predictions_with_know_occ = serie_occ.loc[commun_occ].astype(float)

mae = mean_absolute_error(predictions_with_know_occ, y_occ)
rmse = np.sqrt(mean_squared_error(predictions_with_know_occ, y_occ))
r2   = r2_score(predictions_with_know_occ, y_occ)

print("Nombre de valeurs : " + str(len(predictions_with_know_occ)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

Nombre de valeurs : 8182
R² : 0.627
MAE : 2.33
RMSE : 4.66


In [611]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all_occ,
    mode='lines',
    name='Taux prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final[target_occ],
    mode='lines',
    name="Taux d'occupation réel"
))

fig.update_layout(
    title="Comparaison des taux occupation (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Taux d'occupation (%)",
    hovermode='x unified'
)

fig.show()

<h2>Prédictions sur nouvelles données - 9 au 11 novembre</h2>

In [612]:
df_a_pred = pd.read_csv("../pred j+3/fichier_complet_pour_previsions.csv", sep=";")

features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 
    'force moyenne vent (m/s)', 'jour_semaine', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton','est_rentree'
]

X_a_pred = df_a_pred[features].copy()

y_a_pred = pipe.predict(X_a_pred)

In [613]:
import pandas as pd
import plotly.graph_objects as go

# --- 0) Préparation des données
df_a_pred = df_a_pred.copy()
df_final  = df_final.copy()
df_a_pred['datetime'] = pd.to_datetime(df_a_pred['datetime'])
df_final['datetime']  = pd.to_datetime(df_final['datetime'])

# --- 1) Fenêtres temporelles
deb_2025, fin_2025 = pd.Timestamp('2025-11-09'), pd.Timestamp('2025-11-12')
deb_2024, fin_2024 = pd.Timestamp('2024-11-09'), pd.Timestamp('2024-11-12')

# --- 2) Filtrer les 9–11 novembre
mask_2025 = (df_a_pred['datetime'] >= deb_2025) & (df_a_pred['datetime'] < fin_2025)
mask_2024 = (df_final['datetime']  >= deb_2024) & (df_final['datetime']  < fin_2024)

df_2025 = df_a_pred.loc[mask_2025]
df_2024 = df_final.loc[mask_2024]

# --- 2b) Colonnes de valeurs
y_2025 = y_a_pred[df_2025.index]          # aligne y_a_pred sur df_2025
y_2024 = df_2024['Débit horaire']

# --- 3) Figure et séries principales
fig = go.Figure()

# Courbe 2025 (axe du bas)
fig.add_trace(go.Scatter(
    x=df_2025['datetime'],
    y=y_2025,
    mode='lines',
    name='Débit prédit – 9–11 nov 2025',
    xaxis='x1'
))

# Courbe 2024 (axe du haut)
fig.add_trace(go.Scatter(
    x=df_2024['datetime'],
    y=y_2024,
    mode='lines',
    name='Débit réel – 9–11 nov 2024',
    line=dict(dash='dash'),
    xaxis='x2'
))

# --- 4) Mise en page : deux axes X (dates) superposés
fig.update_layout(
    title='Comparaison des débits – 9–11 novembre (2025 vs 2024)',
    yaxis=dict(title='Débit'),

    # Axe bas = 2025
    xaxis=dict(
        title='Dates 2025 (9–11 nov)',
        range=[deb_2025, fin_2025],
        domain=[0, 1],
        anchor='y',
        rangeslider=dict(visible=False)
    ),

    # Axe haut = 2024
    xaxis2=dict(
        range=[deb_2024, fin_2024],
        overlaying='x',
        side='top',
        rangeslider=dict(visible=False)
    ),

    legend_title='Séries'
)

# --- 5) Référentiel : moyennes horaires toutes années confondues
ref_frames = []

# Historique (ex. 2024)
tmp_hist = df_final[['datetime', 'Débit horaire']].copy()
tmp_hist.rename(columns={'Débit horaire': 'value'}, inplace=True)
ref_frames.append(tmp_hist)

# (optionnel) Inclure aussi les prévisions 2025 dans le référentiel
tmp_pred = df_a_pred[['datetime']].copy()
tmp_pred['value'] = pd.Series(y_a_pred, index=df_a_pred.index).values
ref_frames.append(tmp_pred)

ref_df = pd.concat(ref_frames, ignore_index=True)
ref_df['datetime'] = pd.to_datetime(ref_df['datetime'])
ref_df['weekday']  = ref_df['datetime'].dt.weekday     # lundi=0 ... dimanche=6
ref_df['hour']     = ref_df['datetime'].dt.hour

hourly_means = (
    ref_df
    .groupby(['weekday','hour'])['value']
    .mean()
    .unstack('hour')            # index=0..6 ; colonnes=0..23
)

# --- 6) Utilitaire : tracer un profil moyen (0..23h) projeté sur une date (tiers du graphe)
def tracer_profil_sur_date(fig, date_cible, weekday_for_profile, name, xaxis):
    """
    Projette le profil moyen 'weekday_for_profile' sur la journée 'date_cible' (24 points : 00h..23h)
    et l'ajoute sur l'axe X spécifié (x1 ou x2).
    """
    heures = range(24)
    x_vals = [pd.Timestamp(date_cible).replace(hour=h, minute=0, second=0, microsecond=0) for h in heures]
    y_vals = hourly_means.loc[weekday_for_profile, list(heures)].values

    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode='lines+markers',
        name=name,
        xaxis=xaxis,
        line=dict(dash='dot'),
        hovertemplate="%{x|%d/%m %Hh} – Moyenne: %{y:.2f}<extra></extra>"
    ))

# --- 7) Appliquer la logique des tiers (2025 en bas)
# Dimanche (1er tiers) → 9 novembre 2025
tracer_profil_sur_date(fig, pd.Timestamp("2025-11-09"), 6, "Moyenne horaire – Dimanche (réf.)", xaxis='x1')

# Lundi (2e tiers) → 10 novembre 2025
tracer_profil_sur_date(fig, pd.Timestamp("2025-11-10"), 0, "Moyenne horaire – Lundi (réf.)", xaxis='x1')

# Mardi (3e tiers) → 11 novembre 2025
tracer_profil_sur_date(fig, pd.Timestamp("2025-11-11"), 1, "Moyenne horaire – Mardi (réf.)", xaxis='x1')

# --- (facultatif) Répliquer aussi les profils sur l'axe du haut (2024)
# tracer_profil_sur_date(fig, pd.Timestamp("2024-11-09"), 6, "Moyenne horaire – Dimanche (réf.) [axe 2024]", xaxis='x2')
# tracer_profil_sur_date(fig, pd.Timestamp("2024-11-10"), 0, "Moyenne horaire – Lundi (réf.) [axe 2024]", xaxis='x2')
# tracer_profil_sur_date(fig, pd.Timestamp("2024-11-11"), 1, "Moyenne horaire – Mardi (réf.) [axe 2024]", xaxis='x2')

fig.show()


In [ ]:
import re
import pandas as pd

def add_lag_features(
    df_pred: pd.DataFrame,
    df_hist: pd.DataFrame,
    datetime_col: str,
    value_col: str,
    lag_tags: list,
    exact_match: bool = True,
    asof_tolerance: str = "30min",
    dedup_strategy: str = "mean" 
) -> pd.DataFrame:
    """
    Ajoute des colonnes de lags lag_or_<J>_<H> dans df_pred à partir de df_hist,
    en gérant les timestamps dupliqués côté historique.
    """
    # Copies et typage
    df_pred = df_pred.copy()
    df_hist = df_hist.copy()
    df_pred[datetime_col] = pd.to_datetime(df_pred[datetime_col])
    df_hist[datetime_col] = pd.to_datetime(df_hist[datetime_col])

    # 1) Déduplication de l'historique par horodatage
    if dedup_strategy == "mean":
        hist_unique = (
            df_hist[[datetime_col, value_col]]
            .groupby(datetime_col, as_index=False, sort=True)
            .agg({value_col: "mean"})
        )
    elif dedup_strategy == "first":
        hist_unique = (
            df_hist[[datetime_col, value_col]]
            .sort_values(datetime_col)
            .drop_duplicates(subset=[datetime_col], keep="first")
        )
    elif dedup_strategy == "last":
        hist_unique = (
            df_hist[[datetime_col, value_col]]
            .sort_values(datetime_col)
            .drop_duplicates(subset=[datetime_col], keep="last")
        )
    else:
        raise ValueError("dedup_strategy doit être 'mean', 'first' ou 'last'.")

    # Tri pour merge_asof
    hist_unique = hist_unique.sort_values(datetime_col).reset_index(drop=True)

    # Précompilation regex des tags
    rx = re.compile(r"^lag_or_(\d+)_(\d+)$")

    # 2) Pour chaque tag, calcul du timestamp décalé puis appariement
    for raw_tag in lag_tags:
        tag = raw_tag.strip().strip(",")
        m = rx.match(tag)
        if not m:
            # Tag ignoré si format invalide
            continue

        d_lag = int(m.group(1))  # jours
        h_lag = int(m.group(2))  # heures

        # t_lag = t - J jours - H heures
        lag_dt = df_pred[datetime_col] - pd.to_timedelta(d_lag, unit="D") - pd.to_timedelta(h_lag, unit="H")
        tmp = pd.DataFrame({datetime_col: lag_dt})

        if exact_match:
            merged = tmp.merge(hist_unique, on=datetime_col, how="left")
            df_pred[tag] = merged[value_col].values
        else:
            merged = pd.merge_asof(
                tmp.sort_values(datetime_col).reset_index(drop=True),
                hist_unique,
                left_on=datetime_col,
                right_on=datetime_col,
                direction="nearest",
                tolerance=pd.Timedelta(asof_tolerance)
            )
            
            df_pred[tag] = merged[value_col].values

    return df_pred


In [ ]:
lag_tags = [
    'lag_or_4_0', 'lag_or_4_1', 'lag_or_5_0', 'lag_or_5_1',
    'lag_or_4_23', 'lag_or_6_23', 'lag_or_7_0', 'lag_or_7_1'
]


df_pred_enrichi = add_lag_features(
    df_pred=df_a_pred,          
    df_hist=df_final,           
    datetime_col='datetime',
    value_col='Taux d\'occupation',  
    lag_tags=lag_tags,
    exact_match=True            
)


/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_4745/306704469.py:64: FutureWarning:

'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_4745/306704469.py:64: FutureWarning:

'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_4745/306704469.py:64: FutureWarning:

'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_4745/306704469.py:64: FutureWarning:

'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_4745/306704469.py:64: FutureWarning:

'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_4

In [616]:
features_occ = [
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin',  
    'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton', 'lag_or_4_0', 'lag_or_4_1', 'lag_or_5_0', 'lag_or_5_1', 'lag_or_4_23',
    'lag_or_6_23', 'lag_or_7_0', 'lag_or_7_1', 'est_rentree'
]

Xocc_a_pred = df_pred_enrichi[features_occ].copy()

yocc_a_pred = pipe_occ.predict(Xocc_a_pred)

In [617]:
import pandas as pd
import plotly.graph_objects as go

# --- 0) Préparation des données
df_pred_enrichi = df_pred_enrichi.copy()
df_final  = df_final.copy()
df_pred_enrichi['datetime'] = pd.to_datetime(df_pred_enrichi['datetime'])
df_final['datetime']  = pd.to_datetime(df_final['datetime'])

# --- 1) Fenêtres temporelles
deb_2025, fin_2025 = pd.Timestamp('2025-11-09'), pd.Timestamp('2025-11-12')
deb_2024, fin_2024 = pd.Timestamp('2024-11-09'), pd.Timestamp('2024-11-12')

# --- 2) Filtrer les 9–11 novembre
mask_2025 = (df_pred_enrichi['datetime'] >= deb_2025) & (df_pred_enrichi['datetime'] < fin_2025)
mask_2024 = (df_final['datetime']  >= deb_2024) & (df_final['datetime']  < fin_2024)

df_2025 = df_pred_enrichi.loc[mask_2025]
df_2024 = df_final.loc[mask_2024]

# --- 2b) Colonnes de valeurs
y_2025 = yocc_a_pred[df_2025.index]          # aligne y_a_pred sur df_2025
y_2024 = df_2024['Taux d\'occupation']

# --- 3) Figure et séries principales
fig = go.Figure()

# Courbe 2025 (axe du bas)
fig.add_trace(go.Scatter(
    x=df_2025['datetime'],
    y=y_2025,
    mode='lines',
    name='Taux prédit – 9–11 nov 2025',
    xaxis='x1'
))

# Courbe 2024 (axe du haut)
fig.add_trace(go.Scatter(
    x=df_2024['datetime'],
    y=y_2024,
    mode='lines',
    name='Taux réel – 9–11 nov 2024',
    line=dict(dash='dash'),
    xaxis='x2'
))

# --- 4) Mise en page : deux axes X (dates) superposés
fig.update_layout(
    title='Comparaison des débits – 9–11 novembre (2025 vs 2024)',
    yaxis=dict(title='Débit'),

    # Axe bas = 2025
    xaxis=dict(
        title='Dates 2025 (9–11 nov)',
        range=[deb_2025, fin_2025],
        domain=[0, 1],
        anchor='y',
        rangeslider=dict(visible=False)
    ),

    # Axe haut = 2024
    xaxis2=dict(
        range=[deb_2024, fin_2024],
        overlaying='x',
        side='top',
        rangeslider=dict(visible=False)
    ),

    legend_title='Séries'
)


# --- 5) Référentiel : moyennes horaires toutes années confondues
ref_frames = []

# Historique (ex. 2024)
tmp_hist = df_final[['datetime', 'Taux d\'occupation']].copy()
tmp_hist.rename(columns={'Taux d\'occupation': 'value'}, inplace=True)
ref_frames.append(tmp_hist)

# (optionnel) Inclure aussi les prévisions 2025 dans le référentiel
tmp_pred = df_a_pred[['datetime']].copy()
tmp_pred['value'] = pd.Series(y_a_pred, index=df_a_pred.index).values
ref_frames.append(tmp_pred)

ref_df = pd.concat(ref_frames, ignore_index=True)
ref_df['datetime'] = pd.to_datetime(ref_df['datetime'])
ref_df['weekday']  = ref_df['datetime'].dt.weekday     # lundi=0 ... dimanche=6
ref_df['hour']     = ref_df['datetime'].dt.hour

hourly_means = (
    ref_df
    .groupby(['weekday','hour'])['value']
    .mean()
    .unstack('hour')            # index=0..6 ; colonnes=0..23
)

# --- 6) Utilitaire : tracer un profil moyen (0..23h) projeté sur une date (tiers du graphe)
def tracer_profil_sur_date(fig, date_cible, weekday_for_profile, name, xaxis):
    """
    Projette le profil moyen 'weekday_for_profile' sur la journée 'date_cible' (24 points : 00h..23h)
    et l'ajoute sur l'axe X spécifié (x1 ou x2).
    """
    heures = range(24)
    x_vals = [pd.Timestamp(date_cible).replace(hour=h, minute=0, second=0, microsecond=0) for h in heures]
    y_vals = hourly_means.loc[weekday_for_profile, list(heures)].values

    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode='lines+markers',
        name=name,
        xaxis=xaxis,
        line=dict(dash='dot'),
        hovertemplate="%{x|%d/%m %Hh} – Moyenne: %{y:.2f}<extra></extra>"
    ))

# --- 7) Appliquer la logique des tiers (2025 en bas)
# Dimanche (1er tiers) → 9 novembre 2025
tracer_profil_sur_date(fig, pd.Timestamp("2025-11-09"), 6, "Moyenne horaire – Dimanche (réf.)", xaxis='x1')

# Lundi (2e tiers) → 10 novembre 2025
tracer_profil_sur_date(fig, pd.Timestamp("2025-11-10"), 0, "Moyenne horaire – Lundi (réf.)", xaxis='x1')

# Mardi (3e tiers) → 11 novembre 2025
tracer_profil_sur_date(fig, pd.Timestamp("2025-11-11"), 1, "Moyenne horaire – Mardi (réf.)", xaxis='x1')

# --- (facultatif) Répliquer aussi les profils sur l'axe du haut (2024)
# tracer_profil_sur_date(fig, pd.Timestamp("2024-11-09"), 6, "Moyenne horaire – Dimanche (réf.) [axe 2024]", xaxis='x2')
# tracer_profil_sur_date(fig, pd.Timestamp("2024-11-10"), 0, "Moyenne horaire – Lundi (réf.) [axe 2024]", xaxis='x2')
# tracer_profil_sur_date(fig, pd.Timestamp("2024-11-11"), 1, "Moyenne horaire – Mardi (réf.) [axe 2024]", xaxis='x2')

fig.show()

In [618]:
pred_finale = pd.DataFrame()

nom_arc = ["Champs-Elysées"]* len(df_a_pred['datetime'])

debit = y_a_pred
taux = yocc_a_pred

pred_finale['arc'] = nom_arc

pred_finale['datetime'] = df_a_pred['datetime']

pred_finale['debit_horaire'] = y_a_pred
pred_finale['taux_occupation'] = yocc_a_pred

pred_finale['debit_horaire'] = pred_finale['debit_horaire'].astype(float)
pred_finale['taux_occupation'] = pred_finale['taux_occupation'].astype(float) 

pred_finale.head()

pred_finale.to_csv("predictions_champs_elysees.csv", sep=";")
